In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_mongodb import MongoDBAtlasVectorSearch
from pydantic import Field
import warnings 
import os
from langchain_core.documents import Document
from pymongo import MongoClient
warnings.filterwarnings('ignore')
load_dotenv()

True

In [2]:
len("FepDo-0DrSo")

11

In [3]:
video_id="FepDo-0DrSo"

try:
    fetched_transcript=YouTubeTranscriptApi().fetch(video_id=video_id,languages=['en'])
    # print(fetched_transcript)
    transcript_list=fetched_transcript.to_raw_data()
    # print(transcript_list)
    transcript=''.join(chunk['text'] for chunk in transcript_list)
    # print(transcript)
except TranscriptsDisabled:
    print("No captions avaialble for this ")
except Exception as e:
    print(f"An error occured {e}")

In [4]:
transcript


"Hello all, my name is Kash Nayak andwelcome to my YouTube channel. So guys,today in this particular video, we aregoing to build a rag application withthe help of MongoDB vector search and uhwhenever we talk about rag, it isnothing but retrieval augmentedgeneration. We'll try to see how we canuse MongoDB vector search database andstep by step how we can actually createuh the entire rag application. We'llalso be discussing about the entirearchitecture. Now before I proceed withthis particular video, I'd like to thankMongoDB for sponsoring this particularvideo. So now let's go ahead and startit creating a account in MongoDB. Sofirst of all, you need to go ahead anduh go to this URL www.mongodb.comand then you can go ahead and sign in oryou can also go ahead and click on getstarted. So if you have not if you donot have an account, go ahead and signup. I will go ahead and use my Googleaccount. So let's go ahead and use myGoogle account. So I will just go aheadand use my uh any email id. So

In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
chunks=splitter.split_text(transcript)

In [88]:
len(chunks)

393

In [90]:
chunks.pop(0)

'Hello all, my name is Kash Nayak andwelcome to my YouTube channel. So guys,today in this particular'

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
client=MongoClient(os.getenv("MONGODB_URL"))
client.admin.command('ping')

{'ok': 1}

In [7]:
collection=client['ai_youtube']['rag_embeddings']
# collection.insert_many([{"text":chunk,'embeddings':embeddings.embed_query(chunk)} for chunk in chunks])



In [97]:
query_vector = embeddings.embed_query(
    "Creating embeddings"
)

results = collection.aggregate([
    {
        "$vectorSearch": {
            "index": "youtube_vector_index",
            "path": "embeddings",
            "queryVector": query_vector,
            "numCandidates": 150,
            "limit": 5
        }
    }
])

for doc in results:
    print(doc['text'])

a function to generatethe embeddings. Right? So let's go aheadand create a function called as
the embedding for thisparticular question let's go ahead andsee my embeddings quickly.
you want. And then afterusing this embedding models, so we willgo ahead and create the
it'llgo ahead and create a embeddings now Ican also go ahead and search for what isthis
making sure that it givesyou the embedding right. So once I goahead and do the embedding for


In [98]:
print(collection.full_name)
print(list(collection.list_search_indexes()))

ai_youtube.rag_embeddings
[{'id': '6ad239b90927c4554985c44a', 'name': 'youtube_vector_index', 'type': 'vectorSearch', 'status': 'READY', 'queryable': True, 'latestDefinitionVersion': {'version': 0, 'createdAt': datetime.datetime(2026, 10, 16, 14, 50, 36, 987000)}, 'latestDefinition': {'fields': [{'type': 'vector', 'path': 'embeddings', 'numDimensions': 384, 'similarity': 'cosine'}]}, 'statusDetail': [{'hostname': 'atlas-oeqvq0-shard-00-00', 'status': 'READY', 'queryable': True, 'mainIndex': {'status': 'READY', 'queryable': True, 'definitionVersion': {'version': 0, 'createdAt': datetime.datetime(2026, 10, 16, 14, 50, 36)}, 'definition': {'fields': [{'type': 'vector', 'path': 'embeddings', 'numDimensions': 384, 'similarity': 'cosine'}]}}}, {'hostname': 'atlas-oeqvq0-shard-00-01', 'status': 'READY', 'queryable': True, 'mainIndex': {'status': 'READY', 'queryable': True, 'definitionVersion': {'version': 0, 'createdAt': datetime.datetime(2026, 10, 16, 14, 50, 36)}, 'definition': {'fields': [

In [10]:
llm=init_chat_model(model="openai/gpt-oss-20b",model_provider='groq')

prompt=PromptTemplate(
    template="""
        You are helpful assistant.Answer only from the provided transcript contet. If the context is insufficient , use your some external knowledge too but dont hallucinate and connect it with the video context window
        {context} Question: {question}
    """, input_variables=['context','question'])

vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embeddings,
    
    index_name="youtube_vector_index",
    text_key="text",
    embedding_key="embeddings"
)


In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [12]:
question="which library installation had done in video"
retrieved_docs=retriever.invoke(question)

context_text='\n\n'.join(doc.page_content for doc in retrieved_docs)
final_prompt=prompt.invoke({"context":context_text,"question":question})

In [13]:
retrieved_docs

[Document(id='6ad24204f949e9fc78b1afd7', metadata={'_id': '6ad24204f949e9fc78b1afd7'}, page_content='justgo ahead and install all the librariesthat I specifically want. Okay. So letme go ahead and do'),
 Document(id='6ad24204f949e9fc78b1af7f', metadata={'_id': '6ad24204f949e9fc78b1af7f'}, page_content="withthis particular video, I'd like to thankMongoDB for sponsoring this particularvideo. So now"),
 Document(id='6ad24204f949e9fc78b1afe7', metadata={'_id': '6ad24204f949e9fc78b1afe7'}, page_content='installed. Okay. One more thingthat I need to install is UV add ipikernel so that I can work with'),
 Document(id='6ad24204f949e9fc78b1afe3', metadata={'_id': '6ad24204f949e9fc78b1afe3'}, page_content='open here.So uh we going to use all the specificlibraries uh uh for this and now I willgo ahead and'),
 Document(id='6ad24204f949e9fc78b1b100', metadata={'_id': '6ad24204f949e9fc78b1b100'}, page_content="likethis particular video. This was it formy side. I'll see you in the next video.Have a g

In [142]:
llm.invoke(final_prompt)

AIMessage(content='In the video the speaker installed **`uvicorn`** and **`ipykernel`**.', additional_kwargs={'reasoning_content': 'We need to answer: "Which library installation had done in video?" The transcript: It mentions installing "MongoDB" sponsor? Wait: The transcript: "So with this particular video, I\'d like to thank MongoDB for sponsoring this particular video. So now installed. One more thing that I need to install is UV add ipikernel so that I can work with open here. So uh we going to use all the specific libraries uh uh for this and now I will go ahead and like this particular video."\n\nIt mentions installing "UV add ipikernel". Perhaps they meant "uvicorn"? No, "UV add ipikernel"? Could be "uvicorn" and "ipykernel". The transcript: "UV add ipikernel" maybe "uvicorn + ipykernel"? Actually "UV add ipikernel" maybe "uvicorn" and "ipykernel".\n\nAlso earlier: "I\'d like to thank MongoDB for sponsoring this particular video." That is not a library installation, it\'s a spo

In [143]:
llm.invoke("waht is mongodb atlas")

AIMessage(content='**MongoDB Atlas** is MongoDB’s fully‑managed, cloud‑based database service. It lets you run, scale, and secure MongoDB clusters on the major public cloud platforms—Amazon Web Services (AWS), Microsoft Azure, and Google Cloud Platform (GCP)—without having to manage the underlying infrastructure yourself.\n\n### Key points\n\n| Feature | What it means |\n|---------|---------------|\n| **Fully managed** | Atlas handles installation, patching, backups, monitoring, and scaling. |\n| **Multi‑cloud** | Deploy the same cluster on AWS, Azure, or GCP (or across all three). |\n| **Auto‑scaling** | Cluster size can grow or shrink automatically based on workload. |\n| **High availability** | Replica sets with automatic failover, and multi‑region clusters for global applications. |\n| **Security** | Network isolation, encryption at rest and in transit, IAM integration, audit logs, and fine‑grained access control. |\n| **Performance** | Dedicated resources (CPU, RAM, SSD), index op